In [1]:
# 准备LLM环境

import os
from openai import OpenAI

api_key = os.getenv('DEEPSEEK_API_KEY')

if not api_key:
    raise ValueError('请设置DEEPSEEK_API_KEY')

# 初始化deepseek客户端
client = OpenAI(
    api_key=api_key,
    base_url='https://api.deepseek.com/v1',
)


In [2]:
# 配置System Prompt:定义了Agent的角色、目标及工作方式
SYSTEM_PROMPT = """
你是一个资深的小红书爆款文案专家，擅长结合最新潮流和产品卖点，创作引人入胜、高互动、高转化的笔记文案。

你的任务是根据用户提供的产品和需求，生成包含标题、正文、相关标签和表情符号的完整小红书笔记。

请始终采用'Thought-Action-Observation'模式进行推理和行动。文案风格需活泼、真诚、富有感染力。当完成任务后，请直接输出 strict raw JSON 格式，禁止输出 markdown 或代码块。格式如下：
{
  "title": "小红书标题",
  "body": "小红书正文",
  "hashtags": ["#标签1", "#标签2", "#标签3", "#标签4", "#标签5"],
  "emojis": ["✨", "🔥", "💖"]
}
在生成文案前，请务必先思考并收集足够的信息。
"""

In [3]:
# Tools工具定义： 拓展LLM的能力
TOOLS_DEFINITION = [
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": "搜索互联网上的实时信息，用于获取最新新闻、流行趋势、用户评价、行业报告等。请确保搜索关键词精确，避免宽泛的查询。",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "要搜索的关键词或问题，例如'最新小红书美妆趋势'或'深海蓝藻保湿面膜 用户评价'",
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "query_product_database",
            "description": "查询内部产品数据库，获取指定产品的详细卖点、成分、适用人群、使用方法等信息。",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_name": {
                        "type": "string",
                        "description": "要查询的产品名称，例如'深海蓝藻保湿面膜'",
                    }
                },
                "required": ["product_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_emoji",
            "description": "根据提供的文本内容，生成一组适合小红书风格的表情符号。",
            "parameters": {
                "type": "object",
                "properties": {
                    "context": {
                        "type": "string",
                        "description": "文案的关键内容或情感，例如'惊喜效果'、'补水保湿'",
                    }
                },
                "required": ["context"]
            }
        }
    },  
]

In [4]:
# 准备真实的商品数据，为mock_query_product_database工具提供数据

from glob import glob
text_lines = []

for file_path in glob("product.txt", recursive=True):
    with open(file_path, "r") as file:
        file_text = file.read()

    # 清洗数据：移除空行和前后空格
    text_lines += [
        line.strip() for line in file_text.split("\n") 
        if line.strip()  # 过滤空行
    ]

print(f"商品的内容：'{text_lines}' ")

# 准备Embedding模型
from pymilvus import model as milvus_model

embedding_model = milvus_model.DefaultEmbeddingFunction()

test_embedding = embedding_model.encode_queries(["测试"])
embedding_dim = len(test_embedding[0])
print(f"嵌入维度：{embedding_dim}")

# 创建collection
from pymilvus import MilvusClient
milvus_client = MilvusClient(uri="./milvus_product.db")
collection_name = "product_rag_collection"

if milvus_client.has_collection(collection_name):
    milvus_client.drop_collection(collection_name)

milvus_client.create_collection(
    collection_name=collection_name,
    dimension=embedding_dim, # 维度数
    metric_type="IP", # 内积距离，IP表示值越大越相似
    consistency_level="Strong", #一致性级别：Strong总是读到最新数据，可能稍慢。
)

# 插入数据
from tqdm import tqdm
import re

data = []
cleaned_lines = []
for i, line in enumerate(text_lines):
    # 清理特殊符号和非中文字符
    cleaned_text = re.sub(r"[^\u4e00-\u9fa5a-zA-Z0-9：+()%，. ]", "", line)
    cleaned_lines.append(cleaned_text)

doc_embeddings = embedding_model.encode_documents(text_lines)
for i, line in enumerate(tqdm(text_lines, desc="Creating product embeddings")):
    data.append({"id": i, "vector": doc_embeddings[i], "text": line})
milvus_client.insert(collection_name=collection_name, data=data)

商品的内容：'['益生菌漱口水：15亿CFU口腔益生菌+新西兰麦卢卡蜂蜜，24小时重建口腔微生态。食品级可吞咽配方。规格：250ml', '全自动炒菜机：AI双铲智能翻炒程序，1000+食谱云存储。高温蒸汽自清洁系统。规格：额定功率800W', '空气净化项链：负离子双发射装置，有效清除1米内PM2.5悬浮物。感应呼吸启停技术。规格：钛合金机身', '养发激光帽：148颗医疗级激光灯珠，刺激毛囊新生。可折叠收纳设计。充电式便携型。规格：650nm波长', '水飞蓟素抗氧化妆前乳：SPF30 PA+++，水飞蓟素协同生育酚，妆前阻隔蓝光伤害。含润色微珠光粒子。规格：40ml', '糙米发酵滤液精华水：92%糙米发酵产物滤液，温和促进角质代谢，维稳油敏肌屏障。自来水般清爽质地。规格：160ml', '丝丽动能素导入精华：CT50复合动能素（含透明质酸+矿物质+氨基酸），模拟细胞外基质修复。需配合仪器使用。规格：5ml*6安瓶', '硫磺温泉头皮磨砂膏：死海矿物硫磺+薄荷醇，深层净化毛囊油脂，舒缓头皮瘙痒。磨砂颗粒可溶解。规格：80ml', '巴西莓果冻唇膜：有机巴西莓果浆+乳木果脂，整夜修护唇纹裂口，唤醒粉嫩唇色。果冻状不沾枕。规格：15g', '可食用防晒喷雾：有机覆盆籽油SPF30防晒认证，含玻尿酸锁水膜，全身防护免卸妆。透明无痕质地。规格：100ml', '除螨紫外除菌仪：双波UV-C紫外线灯管+双倍拍打频率，30秒整床杀菌99.9%。便携折叠设计。规格：充电式', '冷萃咖啡滤泡袋：埃塞俄比亚耶加雪菲单一原产地豆，48小时冰滴工艺制作。每袋挂耳现冲。规格：10g×8袋', '磁悬浮扩香仪：静音悬浮涡轮扩散精油微粒，300m²覆盖面积。支持手机智能定时。规格：精油容量150ml', '硅藻土地垫：天然硅藻土高密度压制，3秒超速吸水，防滑纹理底层。浴室专用。规格：40×60cm', '低温煮肉棒：水浴精确温控(55-90℃)，柔性陶瓷涂层不粘内胆，炖煮12小时不断电安全技术。规格：3.5L容量', '乳清蛋白能量棒：三重蛋白矩阵（分离乳清+水解蛋白+酪蛋白），每支25g蛋白0添加糖，即时训练补充。口味：黑巧布朗尼。规格：45g×12支', '智能负重跳绳：4档负重块自由组合（0.5-1.5kg），APP计数脂肪燃烧模式，防缠绕轴承设计。规格：绳长3m可调', '肌肉

/opt/anaconda3/envs/deepseek-p/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


嵌入维度：768


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Creating product embeddings: 100%|██████████| 25/25 [00:00<00:00, 130745.14it/s]


{'insert_count': 25, 'ids': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24], 'cost': 0}

In [5]:
# Tools工具实现：
import random # 用于模拟生成表情
import time # 用于模拟网络延迟
import json

def mock_search_web(query: str) -> str:
    """模拟网页搜索工具，返回预设的搜索结果。"""
    print(f"[Tool Call] 模拟搜索网页：{query}")
    time.sleep(1) # 模拟网络延迟
    if "小红书美妆趋势" in query:
        return "近期小红书美妆流行'多巴胺穿搭'、'早C晚A'护肤理念、'伪素颜'妆容，热门关键词有#氛围感、#抗老、#屏障修复。"
    elif "保湿面膜" in query:
        return "小红书保湿面膜热门话题：沙漠干皮救星、熬夜急救面膜、水光肌养成。用户痛点：卡粉、泛红、紧绷感。"
    elif "深海蓝藻保湿面膜" in query:
        return "关于深海蓝藻保湿面膜的用户评价：普遍反馈补水效果好，吸收快，对敏感肌友好。有用户提到价格略高，但效果值得。"
    else:
        return f"未找到关于 '{query}' 的特定信息，但市场反馈通常关注产品成分、功效和用户体验。"

def mock_query_product_database(product_name: str) -> str:
    """模拟查询产品数据库，返回预设的产品信息。"""
    search_res = milvus_client.search(
        collection_name=collection_name,
        data=embedding_model.encode_queries([product_name]),
        limit=3, #返回最相似的Top3结果
        search_params={
            "metric_type": "IP", 
            "params": {}
        },
        output_fields=["text"]
    )
    retrieved_lines_with_distances = [
      (res["entity"]["text"], res["distance"]) for res in search_res[0]
    ]
    result = "\n".join(
        [line_with_distance[0] for line_with_distance in retrieved_lines_with_distances]
    )
    return result


def mock_generate_emoji(context: str) -> list:
    """模拟生成表情符号，根据上下文提供常用表情。"""
    print(f"[Tool Call] 模拟生成表情符号，上下文：{context}")
    time.sleep(0.2) # 模拟生成延迟
    if "补水" in context or "水润" in context or "保湿" in context:
        return ["💦", "💧", "🌊", "✨"]
    elif "惊喜" in context or "哇塞" in context or "爱了" in context:
        return ["💖", "😍", "🤩", "💯"]
    elif "熬夜" in context or "疲惫" in context:
        return ["😭", "😮‍💨", "😴", "💡"]
    elif "好物" in context or "推荐" in context:
        return ["✅", "👍", "⭐", "🛍️"]
    else:
        return random.sample(["✨", "🔥", "💖", "💯", "🎉", "👍", "🤩", "💧", "🌿"], k=min(5, len(context.split())))

# 将模拟工具函数映射到一个字典，方便通过名称调用
available_tools = {
    "search_web": mock_search_web,
    "query_product_database": mock_query_product_database,
    "generate_emoji": mock_generate_emoji,
}

In [6]:
# 构建DS Agent工作流，通过循环来模拟Tought-Action-Observation过程
import json
import re

def generate_rednote(product_name: str, tone_style: str = "活泼甜美", max_iterations: int = 5) -> str:
    """
    使用 DeepSeek Agent 生成小红书爆款文案。
    
    Args:
        product_name (str): 要生成文案的产品名称。
        tone_style (str): 文案的语气和风格，如"活泼甜美"、"知性"、"搞怪"等。
        max_iterations (int): Agent 最大迭代次数，防止无限循环。
        
    Returns:
        str: 生成的爆款文案（JSON 格式字符串）。
    """

    print(f"\n🚀 启动小红书文案生成助手，产品：{product_name}，风格：{tone_style}\n")

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"请为产品「{product_name}」生成一篇小红书爆款文案。要求：语气{tone_style}，包含标题、正文、至少5个相关标签和5个表情符号。请以完整的JSON格式输出，并确保JSON内容用markdown代码块包裹（例如：```json{{...}}```）。"}
    ]

    iteration_count = 0
    final_response = None

    while iteration_count < max_iterations:
        iteration_count += 1
        print(f"-- Iteration {iteration_count} --")

        try:
            # 调用DeepSeek API，传入对话历史和工具定义
            response = client.chat.completions.create(
                model="deepseek-chat",
                messages=messages,
                tools=TOOLS_DEFINITION, # 告知模型可用的工具
                tool_choice="auto", # 允许模型自动决定是否使用工具
                response_format={"type": "json_object"},
            )

            response_message = response.choices[0].message
            print(f"LLM返回结果： '{response_message}'")

            if response_message.tool_calls: # 模型决定调用工具
                print(f"Agent: 决定调用工具")
                messages.append(response_message) 

                tool_outputs = []

                # 根据模型返回的工具列表， 执行可用的工具
                for tool_call in response_message.tool_calls:
                    function_name = tool_call.function.name
                    function_args = json.loads(tool_call.function.arguments) if tool_call.function.arguments else {}

                    print(f"Agent Action: 调用工具 '{function_name}', 参数：'{function_args}', 原始参数：'{tool_call.function.arguments}' ")

                    if function_name in available_tools:
                        tool_function = available_tools[function_name]
                        tool_result = tool_function(**function_args)
                        print(f"Observation: 工具返回结果： '{tool_result}'")

                        tool_outputs.append({
                            "role": "tool",
                            "tool_call_id": tool_call.id,
                            "content": str(tool_result)
                        })
                    else:
                        error_message = f"错误：未知的工具 '{function_name}'"
                        print(error_message)
                        tool_outputs.append({
                            "tool_call_id": tool_call.id,
                            "role": "tool",
                            "content": error_message
                        })
                messages.extend(tool_outputs)

            elif response_message.content: #模型返回的是内容，通常是最终答案
                print(f"[模型生成结果] {response_message.content}")

            
                try:
                    final_response = json.loads(response_message.content)
                    print("Agent: 任务完成，直接解析最终JSON文案。")
                    return json.dumps(final_response, ensure_ascii=False, indent=2)
                except json.JSONDecodeError:
                    print("Agent: 生成了非JSON格式内容或非Markdown JSON块，可能还在思考或出错。")
                    messages.append(response_message) # 非JSON格式，继续对话
                # --- END: 添加 JSON 提取和解析逻辑 ---

            else:
                print("Agent: 未知响应，可能需要更多交互。")
                break 
            
        except Exception as e:
            print(f"调用DeepSeek API 时发生错误: {e}")
            break
        

    print("\n⚠️ Agent 达到最大迭代次数或未能生成最终文案。请检查Prompt或增加迭代次数。")
    return "未能成功生成文案。"


In [7]:
# 测试文案生成1

product_name_1 = "蓝光美牙仪"
tone_style_1 = "专业认真"
result_1 = generate_rednote(product_name_1, tone_style_1)

print("\n--- 生成的文案 1 ---")
print(result_1)


🚀 启动小红书文案生成助手，产品：蓝光美牙仪，风格：专业认真

-- Iteration 1 --
LLM返回结果： 'ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_0_d80a5cc2-fbc5-4099-9820-549c92862a19', function=Function(arguments='{"product_name":"蓝光美牙仪"}', name='query_product_database'), type='function', index=0)])'
Agent: 决定调用工具
Agent Action: 调用工具 'query_product_database', 参数：'{'product_name': '蓝光美牙仪'}', 原始参数：'{"product_name":"蓝光美牙仪"}' 
Observation: 工具返回结果： '茶树精油祛痘贴：食品级水胶体敷料+澳洲茶树精油，快速吸出白头脓包，隐形防菌屏障。圆形昼夜两用装。规格：24贴
硅藻土地垫：天然硅藻土高密度压制，3秒超速吸水，防滑纹理底层。浴室专用。规格：40×60cm
巴西莓果冻唇膜：有机巴西莓果浆+乳木果脂，整夜修护唇纹裂口，唤醒粉嫩唇色。果冻状不沾枕。规格：15g'
-- Iteration 2 --
LLM返回结果： 'ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_0_b8ec4199-dd33-4412-9c10-6e9372233c61', function=Function(arguments='{"query":"蓝光美牙仪 用户评价"}', name='search_web'), type

In [8]:
# 测试文案生成1

product_name_2 = "除菌仪"
tone_style_2 = "知性优雅"
result_2 = generate_rednote(product_name_2, tone_style_2)

print("\n--- 生成的文案 2 ---")
print(result_2)


🚀 启动小红书文案生成助手，产品：除菌仪，风格：知性优雅

-- Iteration 1 --
LLM返回结果： 'ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_0_e1f80881-cfbb-4dd0-84f4-b1814f820772', function=Function(arguments='{"product_name":"除菌仪"}', name='query_product_database'), type='function', index=0)])'
Agent: 决定调用工具
Agent Action: 调用工具 'query_product_database', 参数：'{'product_name': '除菌仪'}', 原始参数：'{"product_name":"除菌仪"}' 
Observation: 工具返回结果： '茶树精油祛痘贴：食品级水胶体敷料+澳洲茶树精油，快速吸出白头脓包，隐形防菌屏障。圆形昼夜两用装。规格：24贴
硅藻土地垫：天然硅藻土高密度压制，3秒超速吸水，防滑纹理底层。浴室专用。规格：40×60cm
巴西莓果冻唇膜：有机巴西莓果浆+乳木果脂，整夜修护唇纹裂口，唤醒粉嫩唇色。果冻状不沾枕。规格：15g'
-- Iteration 2 --
LLM返回结果： 'ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_0_47c49079-cb8a-4c10-8af2-c73de6c7fb88', function=Function(arguments='{"context":"除菌仪"}', name='generate_emoji'), type='functio

In [9]:
# 增加格式化函数，解析JSON字符串
import json

def format_rednote_for_markdown(json_string: str) -> str:
    """
    将 JSON 格式的小红书文案转换为 Markdown 格式，以便于阅读和发布。

    Args:
        json_string (str): 包含小红书文案的 JSON 字符串。
                           预计格式为 {"title": "...", "body": "...", "hashtags": [...], "emojis": [...]}

    Returns:
        str: 格式化后的 Markdown 文本。
    """
    try:
        data = json.loads(json_string)
    except json.JSONDecodeError as e:
        return f"错误：无法解析 JSON 字符串 - {e}\n原始字符串：\n{json_string}"

    title = data.get("title", "无标题")
    body = data.get("body", "")
    hashtags = data.get("hashtags", [])
    # 表情符号通常已经融入标题和正文中，这里可以选择是否单独列出
    # emojis = data.get("emojis", []) 

    # 构建 Markdown 文本
    markdown_output = f"## {title}\n\n" # 标题使用二级标题
    
    # 正文，保留换行符
    markdown_output += f"{body}\n\n"
    
    # Hashtags
    if hashtags:
        hashtag_string = " ".join(hashtags) # 小红书标签通常是空格分隔
        markdown_output += f"{hashtag_string}\n"
        
    # 如果需要，可以单独列出表情符号，但通常它们已经包含在标题和正文中
    # if emojis:
    #     emoji_string = " ".join(emojis)
    #     markdown_output += f"\n使用的表情：{emoji_string}\n"
        
    return markdown_output.strip() # 去除末尾多余的空白

In [14]:
generated_json_output = """
{\n  "title": "💯 居家必备！这款除菌仪让生活更安心，细菌无处藏身！",\n  "body": "最近入手了这款超实用的除菌仪，简直是居家生活的救星！✨ 无论是厨房、卫生间还是宝宝的玩具，轻轻一照，细菌瞬间消失无踪，再也不用担心细菌滋生啦！\\n\\n🌟 卖点速递：\\n1️⃣ 高效除菌：99.9%的细菌都能被消灭，安全又放心。\\n2️⃣ 便携设计：小巧轻便，随时随地都能使用。\\n3️⃣ 一键操作：简单易用，老人小孩都能轻松上手。\\n4️⃣ 环保节能：低耗电量，绿色生活从细节开始。\\n\\n用了它之后，家里的卫生状况明显提升，连空气都感觉清新了许多！💖 真心推荐给每一个注重健康生活的你～",\n  "hashtags": ["#居家必备", "#除菌神器", "#健康生活", "#家电推荐", "#生活好物"],\n  "emojis": ["💯", "✨", "🌟", "💖", "1️⃣"]\n}
"""

markdown_note = format_rednote_for_markdown(generated_json_output)

# 打印结果
print("--- 格式化后的小红书文案 (Markdown) ---")
print(markdown_note)

--- 格式化后的小红书文案 (Markdown) ---
## 💯 居家必备！这款除菌仪让生活更安心，细菌无处藏身！

最近入手了这款超实用的除菌仪，简直是居家生活的救星！✨ 无论是厨房、卫生间还是宝宝的玩具，轻轻一照，细菌瞬间消失无踪，再也不用担心细菌滋生啦！

🌟 卖点速递：
1️⃣ 高效除菌：99.9%的细菌都能被消灭，安全又放心。
2️⃣ 便携设计：小巧轻便，随时随地都能使用。
3️⃣ 一键操作：简单易用，老人小孩都能轻松上手。
4️⃣ 环保节能：低耗电量，绿色生活从细节开始。

用了它之后，家里的卫生状况明显提升，连空气都感觉清新了许多！💖 真心推荐给每一个注重健康生活的你～

#居家必备 #除菌神器 #健康生活 #家电推荐 #生活好物
